# Retail Sales Exploratory Analysis

**A reproducible, evidence-led study of transaction patterns and customer activity**

This notebook moves from data validation to statistical interpretation. Its purpose is not merely to produce charts, but to show which conclusions the data supports, how missing values are handled, and where the evidence is limited.

### Questions

1. Is the data complete and internally consistent?
2. What does the transaction-value distribution reveal?
3. How does recorded sales activity vary across the month?
4. Which customers contribute the most sales?
5. How is daily transaction volume related to daily sales?
6. What should decision-makers *not* infer from this small dataset?


## 1. Environment setup

The setup cell works in two situations:

- **Google Colab:** it clones the repository automatically.
- **Local Jupyter:** it locates the project root from the current directory.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = "https://github.com/DrHiraBenish/sales-data-analysis.git"

if "google.colab" in sys.modules:
    project_root = Path("/content/sales-data-analysis")
    if not project_root.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(project_root)], check=True)
    os.chdir(project_root)
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    project_root = next(
        (path for path in candidates if (path / "data" / "grocery_sales.csv").exists()),
        None,
    )
    if project_root is None:
        raise FileNotFoundError("Run this notebook from the repository or notebooks folder.")
    os.chdir(project_root)

sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.analyze_sales import (
    bootstrap_interval,
    calculate_metrics,
    configure_plot_style,
    load_and_validate,
    prepare_analysis_tables,
)

configure_plot_style()
print(f"Project root: {project_root}")


## 2. Load and audit the data

The validation function checks required fields, date parsing, unique transaction IDs, missing customer IDs, and non-positive sales values. These checks make assumptions explicit instead of allowing silent data-quality problems to flow into the analysis.


In [ ]:
data_path = project_root / "data" / "grocery_sales.csv"
raw = load_and_validate(data_path)

audit = pd.DataFrame(
    {
        "Check": [
            "Rows",
            "Columns",
            "Duplicate rows",
            "Duplicate transaction IDs",
            "Missing sales values",
            "Date range",
        ],
        "Result": [
            len(raw),
            raw.shape[1],
            raw.duplicated().sum(),
            raw["transaction_id"].duplicated().sum(),
            raw["sales"].isna().sum(),
            f"{raw['transaction_date'].min().date()} to {raw['transaction_date'].max().date()}",
        ],
    }
)
audit


In [ ]:
raw.head()

### Missing-value decision

Four of 281 records (1.4%) have no sales amount. These records are excluded from calculations involving sales. Mean imputation is deliberately avoided because it would create revenue that was never observed. The missing records remain available for audit, and the original dataset is never overwritten.


In [ ]:
clean, daily, customer = prepare_analysis_tables(raw)
metrics = calculate_metrics(raw, clean, daily, customer)

print(f"Original records: {len(raw)}")
print(f"Complete sales records: {len(clean)}")
print(f"Excluded from revenue calculations: {raw['sales'].isna().sum()}")

raw.loc[raw["sales"].isna()]


## 3. Core performance indicators

In [ ]:
kpis = pd.DataFrame(
    {
        "Metric": [
            "Total recorded sales",
            "Transactions analysed",
            "Unique customers",
            "Mean transaction",
            "Median transaction",
            "Highest daily sales",
        ],
        "Value": [
            f"{metrics['total_sales']:,.2f}",
            f"{metrics['records_analyzed']:,}",
            f"{metrics['unique_customers']:,}",
            f"{metrics['mean_transaction']:,.2f}",
            f"{metrics['median_transaction']:,.2f}",
            f"{metrics['highest_daily_sales']:,.2f}",
        ],
    }
)
kpis


## 4. Transaction-value distribution

![Transaction-value distribution](../reports/figures/sales_distribution.svg)

The mean (**56.13**) is higher than the median (**36.28**), showing a right-skewed distribution. Six transactions exceed the IQR upper fence of **185.36**. They are retained: the dataset provides no product or invoice detail that would justify classifying them as errors.

These high-value observations represent **7.6%** of total recorded sales. Reporting both mean and median prevents a small upper tail from defining the “typical” transaction.


In [ ]:
q1, q3 = clean["sales"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
high_value_transactions = clean.loc[clean["sales"] > upper_fence]

fig, axes = plt.subplots(
    2, 1, figsize=(9, 6.5), gridspec_kw={"height_ratios": [4, 1]}
)
sns.histplot(clean, x="sales", bins=20, kde=True, color="#2F80ED", ax=axes[0])
axes[0].axvline(clean["sales"].median(), color="#19A7A0", linestyle="--", label="Median")
axes[0].axvline(clean["sales"].mean(), color="#F2994A", linestyle="--", label="Mean")
axes[0].set(title="Transaction-value distribution", xlabel="Sales amount")
axes[0].legend(frameon=False)
sns.boxplot(data=clean, x="sales", color="#EAF2F8", ax=axes[1])
axes[1].set(xlabel="Sales amount", ylabel="")
plt.tight_layout()
plt.show()

high_value_transactions


### Uncertainty around the centre

A percentile bootstrap provides a transparent estimate of sampling uncertainty without assuming that the strongly skewed transaction values follow a normal distribution.


In [ ]:
intervals = pd.DataFrame(
    {
        "Statistic": ["Mean", "Median"],
        "Estimate": [clean["sales"].mean(), clean["sales"].median()],
        "Bootstrap 95% interval": [
            bootstrap_interval(clean["sales"], "mean"),
            bootstrap_interval(clean["sales"], "median"),
        ],
    }
)
intervals


## 5. Daily sales activity

![Daily recorded sales](../reports/figures/daily_sales.svg)

The highest recorded day is **22 September (1,147.62)** and the lowest is **9 September (119.70)**. The range is substantial, but one month is not enough to establish seasonality or a durable weekday pattern. These values describe the observed month only.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(daily["transaction_date"], daily["total_sales"], color="#2F80ED", marker="o")
ax.set(
    title="Daily recorded sales — September 2020",
    xlabel="Date",
    ylabel="Total sales",
)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

daily.nlargest(5, "total_sales")


## 6. Customer contribution

![Top customers](../reports/figures/top_customers.svg)

Customer 48 has the highest recorded sales (**524.41 across four transactions**). The top 10 customers contribute **26.3%** of total sales, so the observed month is not dominated by one customer.

Within the available month, **74.4%** of customers made more than one transaction and generated **87.0%** of sales. This measures repeat activity *within September*; it is not a long-term retention rate.


In [ ]:
top_10 = customer.head(10)
top_10[["customer_id", "total_sales", "transactions", "average_transaction"]]


In [ ]:
activity_summary = (
    customer.groupby("activity_group", as_index=False)
    .agg(customers=("customer_id", "nunique"), total_sales=("total_sales", "sum"))
)
activity_summary["customer_share_percent"] = (
    activity_summary["customers"] / activity_summary["customers"].sum() * 100
)
activity_summary["sales_share_percent"] = (
    activity_summary["total_sales"] / activity_summary["total_sales"].sum() * 100
)
activity_summary.round(1)


## 7. Daily sales and transaction volume

![Sales and transaction volume](../reports/figures/sales_vs_volume.svg)

Daily transaction count and daily total sales have a Pearson correlation of **0.694**. This is a moderately strong positive association, but it requires cautious interpretation: daily sales is the sum of transaction values, so transaction count is mechanically part of the outcome. The analysis does not establish causation.


In [ ]:
correlation = daily["total_sales"].corr(daily["transactions"])

fig, ax = plt.subplots(figsize=(8.5, 5.8))
sns.regplot(
    data=daily,
    x="transactions",
    y="total_sales",
    scatter_kws={"s": 55, "alpha": 0.8, "color": "#7257D5"},
    line_kws={"color": "#F2994A"},
    ax=ax,
)
ax.set(
    title=f"Daily sales and transaction volume (r = {correlation:.3f})",
    xlabel="Transactions per day",
    ylabel="Total daily sales",
)
plt.tight_layout()
plt.show()


## 8. Conclusions

1. The usable dataset contains **277 complete transactions** totalling **15,547.35**.
2. Transaction values are right-skewed, so the **median** should accompany the mean when reporting typical purchase size.
3. Six high-value transactions are influential but not extreme enough to dominate total sales.
4. Daily recorded sales vary considerably; however, a single month cannot support seasonal or forecasting claims.
5. Repeat-within-month customers contribute most of the observed sales, but longer history is needed to measure retention.
6. Transaction volume is associated with total daily sales, although the relationship is partly structural and not causal evidence.

### Limitations

- Only one month of data is available.
- The source, sampling method, and currency were not documented in the earlier repository.
- Product, category, location, cost, and profit data are unavailable.
- Customer behaviour cannot be followed beyond September 2020.
- The sample is suitable for demonstrating EDA, not for generalising to a retail population.

### Recommended next data collection

For a decision-grade analysis, collect at least 12 months of transactions with product/category, store/channel, quantity, price, discount, cost, and customer-acquisition fields. That would support seasonality, cohort retention, profitability, segmentation, and forecasting.
